
# ML: Predict and reduce 30 Day Readmissions

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/hls/patient-readmission/hls-patient-readmision-flow-4.png" style="float: right; margin-left: 30px; margin-top:10px" width="650px" />

We now have our data cleaned and secured. We saw how to create and analyze our first patient cohorts.

Let's now take it to the next level and start building a Machine Learning model to predict wich patients are at risk. 

We'll then be able to explain the model at a statistical level, understanding which features increase the readmission risk. This information will be critical in how we can specialize care for a specific patient population.


<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F04-Data-Science-ML%2F04.1-Feature-Engineering-patient-readmission&demo_name=lakehouse-hls-readmission&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-hls-readmission%2F04-Data-Science-ML%2F04.1-Feature-Engineering-patient-readmission&version=1">

In [0]:
%pip install mlflow==2.19.0
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/27.4 MB ? eta -:--:--
   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.6/27.4 MB 18.8 MB/s eta 0:00:02
   ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/27.4 MB 114.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━ 17.0/27.4 MB 272.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━ 25.4/27.4 MB 250.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 27.4/27.4 MB 255.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/5.9 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 5.8/5.9 MB 295.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 162.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/147.8 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/114.9 kB ? eta -:--:--
   ━━

In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_hls_readmission`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.



# 1/ Building our Features for Patient readmission analysis

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/hls/patient-readmission/patient-risk-ds-flow-1.png?raw=true" width="700px" style="float: right; margin-left: 10px;" />
Our first step is now to merge our different tables adding extra features to be able to train our model.

We will use the `encounters_readmissions` table as the label we want to predict:

is there a readmission within 30 days.

In [0]:
from pyspark.sql import Window
# Let's create our label: we'll predict the  30 days readmission risk
windowSpec = Window.partitionBy("PATIENT").orderBy("START")
labels = spark.table('encounters').select("PATIENT", "Id", "START", "STOP") \
              .withColumn('30_DAY_READMISSION', F.when(col('START').cast('long') - F.lag(col('STOP')).over(windowSpec).cast('long') < 30*24*60*60, 1).otherwise(0))
display(labels)

PATIENT,Id,START,STOP,30_DAY_READMISSION
004f14e7-dc30-71bb-c934-50b5c632ed9b,9f9f98cb-c55c-2255-8adb-28e189ef1ffa,2010-06-26T19:27:33Z,2010-06-26T19:49:27Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,8b78cd72-a535-43ed-4c39-ac21b9f546a7,2011-12-17T19:27:33Z,2011-12-17T19:52:54Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,4295c416-05e9-63c5-5ded-7534d0f03740,2013-05-13T23:10:30Z,2013-05-13T23:25:30Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,0975986f-90bd-8539-c9b3-7ae6c6f5a590,2013-07-13T19:27:33Z,2013-07-13T19:42:33Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,6f1de637-751c-c3bf-10e1-612571f4f338,2014-07-19T19:27:33Z,2014-07-19T20:10:46Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,20fbd2d7-72f5-5309-c652-d59f1e07627c,2015-05-03T23:10:30Z,2015-05-03T23:30:21Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,6248ba13-74da-4965-cff8-d23d9f35a402,2015-07-25T19:27:33Z,2015-07-25T20:21:00Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,9a90ea6d-49ea-59ec-8b08-c18510c8ad09,2016-05-28T19:27:33Z,2016-05-28T20:10:36Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,4fa253d7-059a-7065-15c8-e5ce88870b2d,2016-07-30T19:27:33Z,2016-07-30T20:37:30Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,9058fbbd-984f-4176-de72-9e4ac85a73be,2016-09-17T19:27:33Z,2016-09-17T19:42:33Z,0


### Join readmission information with patient features

Let's add features describing our patients cohort. 

In this case we're doing some one hot encoding leveraging `get_dummies` from the Pandas API on Spark to transform categories into vector our model will be able to process.

### Optional: leverage Databricks Feature Store

Being able to save our features in dedicated feature store simplify data management cross teamn, allowing features to be shared but also used in real-time leveraging realtime feature serving (automatically backed by ELTP databases).

To keep this notebook simple, we won't be using the Feature store. If you are interested, open the [03.6-Feature-Store-ML-patient-readmission](/advanced-feature-store/03.6-Feature-Store-ML-patient-readmission) for a complete example.

In [0]:
import pyspark.pandas as ps

# Define Patient Features logic
def compute_pat_features(data):
  data = data.pandas_api()
  data = ps.get_dummies(data, columns=['MARITAL', 'RACE', 'ETHNICITY', 'GENDER'],dtype = 'int64').to_spark()
  return data

In [0]:
cohort_name = 'COVID-19-cohort' #or could be all_patients
cohort = spark.sql(f"SELECT p.* FROM cohort c INNER JOIN patients p on c.patient=p.id WHERE c.name='{cohort_name}'") \
              .dropDuplicates(["id"])
cohort_features_df = compute_pat_features(cohort)
cohort_features_df.display()

Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,MAIDEN,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME,_rescued_data,MARITAL_D,MARITAL_M,MARITAL_S,MARITAL_W,RACE_asian,RACE_black,RACE_hawaiian,RACE_native,RACE_other,RACE_white,ETHNICITY_hispanic,ETHNICITY_nonhispanic,GENDER_F,GENDER_M
0041801c-0da3-70c3-5daf-a24b26454f48,2006-02-03,null,999-99-7505,S99944023,null,null,Tonia627,Runte676,null,null,Haverhill Massachusetts US,557 Hansen Spur,Middleborough,Massachusetts,Plymouth County,null,0,41.923230711923054,-70.86910617094345,20021.38,4239.21,55775,null,0,0,0,0,0,0,0,0,0,1,0,1,1,0
005430cb-2762-4138-6d9a-faa31b462ca0,2018-03-01,null,999-67-4871,null,null,null,Everett935,Spinka232,null,null,Springfield Massachusetts US,899 Koss Knoll,Andover,Massachusetts,Essex County,25009.0,1810,42.60172529131515,-71.20954240577458,16883.04,5024.34,150489,null,0,0,0,0,0,0,0,0,0,1,0,1,0,1
00aa4fa0-2e2d-c63c-7d10-21a4242921de,1961-07-27,null,999-64-9880,S99943948,X25587660X,Mr.,Man114,Wuckert783,null,null,Sandwich Massachusetts US,888 Osinski Throughway,Somerset,Massachusetts,Bristol County,25005.0,2725,41.71678990260496,-71.22103262868447,132780.43,0.0,71035,null,0,0,1,0,0,0,0,0,0,1,0,1,0,1
00b55b42-527e-d8b8-24f5-5323eb380a17,2004-07-26,null,999-32-5532,S99931320,null,Ms.,Romelia905,Harber290,null,null,Weymouth Massachusetts US,833 Osinski Mill,Boston,Massachusetts,Suffolk County,25025.0,2109,42.303333981024736,-71.12175002894712,3350.0,23691.23,16957,null,0,0,0,0,0,0,0,0,0,1,0,1,1,0
00bf5a03-284c-1c08-2a63-7f755314a409,2002-11-18,null,999-63-3181,S99912424,X50819094X,Ms.,Ailene520,Morissette863,null,null,Boston Massachusetts US,730 Jaskolski Parade,Fall River,Massachusetts,Bristol County,25005.0,2723,41.68412665686501,-71.19130747787655,22203.27,742455.44,26961,null,0,0,0,0,0,0,0,0,0,1,0,1,1,0
00d43881-d7cc-82b7-1f22-45f5718b3951,2013-08-03,null,999-85-2951,null,null,null,Rebbecca977,Kuphal363,null,null,Northborough Massachusetts US,331 Maggio Approach,Shrewsbury,Massachusetts,Worcester County,null,0,42.28257905716658,-71.73532912930618,10616.33,5272.02,86306,null,0,0,0,0,0,1,0,0,0,0,0,1,1,0
00f313b1-fee7-71b6-c3ff-0cf3620b1667,1993-11-10,null,999-24-7849,S99924297,X26827684X,Mr.,Otis335,Price929,null,null,Incheon Incheon KR,689 Gutmann Meadow Apt 57,Needham,Massachusetts,Norfolk County,25021.0,2494,42.318080066623565,-71.22414256667504,44485.5,2229.19,207991,null,0,1,0,0,1,0,0,0,0,0,0,1,0,1
010149cd-ed06-b9b8-3959-32352e0a093b,1948-10-24,null,999-89-4864,S99946186,X36703893X,Mrs.,Samira471,Thiel172,null,Gulgowski816,Haverhill Massachusetts US,442 Mertz Mill,Mattapoisett,Massachusetts,Plymouth County,null,0,41.680574647660734,-70.76274511814998,541413.72,275220.2,64878,null,0,1,0,0,0,0,0,0,0,1,0,1,1,0
011b3de4-afe0-0604-fdd6-8e97b5f118f2,2010-09-07,null,999-64-4460,null,null,null,Dionna991,White193,null,null,Kingston Massachusetts US,419 Davis Heights Apt 75,Wilbraham,Massachusetts,Hampden County,25013.0,1095,42.17383118077709,-72.45124606915937,2450.0,105725.23,15663,null,0,0,0,0,0,0,0,0,0,1,1,0,1,0
015f695a-9ac9-5c91-6b24-8dee567bbf15,1984-09-04,null,999-33-6208,S99999640,X82888578X,Mrs.,Helaine42,Runolfsdottir785,null,Okuneva707,Salem Massachusetts US,599 Pfeffer Terrace Unit 37,Needham,Massachusetts,Norfolk County,25021.0,2492,42.3080554148418,-71.28729908748122,128480.99,802529.12,27524,null,1,0,0,0,0,0,0,0,0,1,0,1,1,0


In [0]:
def compute_enc_features(data):
  data = data.dropDuplicates(["Id"])
  data = data.withColumn('enc_length', F.unix_timestamp(col('stop'))- F.unix_timestamp(col('start')))
  data = data.pandas_api()
#   return data
  data = ps.get_dummies(data, columns=['ENCOUNTERCLASS'],dtype = 'int64').to_spark()
  
  return (
    data
    .select(
      col('Id').alias('ENCOUNTER_ID'),
      'BASE_ENCOUNTER_COST',
      'TOTAL_CLAIM_COST',
      'PAYER_COVERAGE',
      'enc_length',
      'ENCOUNTERCLASS_ambulatory',
      'ENCOUNTERCLASS_emergency',
      'ENCOUNTERCLASS_hospice',
      'ENCOUNTERCLASS_inpatient',
      'ENCOUNTERCLASS_outpatient',
      'ENCOUNTERCLASS_wellness',
    )
  )
enc_features_df = compute_enc_features(spark.table('encounters'))
display(enc_features_df)

ENCOUNTER_ID,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,enc_length,ENCOUNTERCLASS_ambulatory,ENCOUNTERCLASS_emergency,ENCOUNTERCLASS_hospice,ENCOUNTERCLASS_inpatient,ENCOUNTERCLASS_outpatient,ENCOUNTERCLASS_wellness
00001222-4aee-9c3e-ce88-babe49bf125c,85.55,234.71,234.71,900,1,0,0,0,0,0
000098b9-8f40-946b-7835-f6b6626db72f,85.55,805.43,770.43,12000,1,0,0,0,0,0
0000fa06-a2b7-7cde-6133-7423e26d2b86,85.55,1253.11,0.0,1918,0,0,0,0,1,0
00011a52-da46-709e-be20-50f61f54d4b2,110.92,19947.63,15711.48,2333700,0,0,0,0,0,0
00014668-e97a-20b5-eca2-be8115d3d896,85.55,952.4,902.4,13560,1,0,0,0,0,0
000174d8-4d45-1b44-c42d-05cb8cb225ef,85.55,516.95,481.95,5225,1,0,0,0,0,0
0003d5b2-8214-abe1-d991-eb26cadf01ef,142.58,1652.26,1321.78,2295,0,0,0,0,0,0
0003e61d-9662-5111-f6ad-48646551f28c,85.55,802.11,0.0,2908,0,0,0,0,1,0
000407f0-5cbd-ff9d-c73f-568d8577de08,146.18,1008.98,0.0,3600,0,1,0,0,0,0
00041427-7a7b-9927-cea0-2aac295cb576,85.55,740.69,705.69,3243,0,0,0,0,1,0


In [0]:
enc_features_df = compute_enc_features(spark.table('encounters'))
training_dataset = cohort_features_df.join(labels, [labels.PATIENT==cohort_features_df.Id], "inner") \
                                     .join(enc_features_df, [labels.Id==enc_features_df.ENCOUNTER_ID], "inner") \
                                     .drop("Id", "_rescued_data", "SSN", "DRIVERS", "PASSPORT", "FIRST", "LAST", "ADDRESS", "BIRTHPLACE")
### Adding extra feature such as patient age at encounter
training_dataset = training_dataset.withColumnRenamed("PATIENT", "patient_id") \
                                   .withColumn("age_at_encounter", ((F.datediff(col('START'), col('BIRTHDATE'))) / 365.25))

training_dataset.write.mode('overwrite').saveAsTable("training_dataset")
display(spark.table("training_dataset"))

BIRTHDATE,DEATHDATE,PREFIX,SUFFIX,MAIDEN,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME,MARITAL_D,MARITAL_M,MARITAL_S,MARITAL_W,RACE_asian,RACE_black,RACE_hawaiian,RACE_native,RACE_other,RACE_white,ETHNICITY_hispanic,ETHNICITY_nonhispanic,GENDER_F,GENDER_M,patient_id,START,STOP,30_DAY_READMISSION,ENCOUNTER_ID,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,enc_length,ENCOUNTERCLASS_ambulatory,ENCOUNTERCLASS_emergency,ENCOUNTERCLASS_hospice,ENCOUNTERCLASS_inpatient,ENCOUNTERCLASS_outpatient,ENCOUNTERCLASS_wellness,age_at_encounter
1960-10-12,2020-11-06,Mr.,null,null,Fall River,Massachusetts,Bristol County,25005.0,2747,41.750834597751016,-71.0452961077422,162261.25,895558.02,30440,0,0,1,0,0,0,0,0,0,1,0,1,0,1,5646f81d-85f2-c2d2-aa1d-3c30a0413148,2002-04-24T03:33:13Z,2002-04-24T04:21:41Z,0,0003e61d-9662-5111-f6ad-48646551f28c,85.55,802.11,0.0,2908,0,0,0,0,1,0,41.530458590006845
1949-11-14,null,Ms.,null,null,Bolton,Massachusetts,Worcester County,null,0,42.42686500765992,-71.61398510432554,259195.31,1819638.61,817966,0,0,1,0,0,1,0,0,0,0,0,1,1,0,ec3eb7ce-a707-49d3-c67b-758d354b1927,1949-12-19T02:05:57Z,1949-12-19T02:20:57Z,0,0005e160-77df-b17d-7d70-8d34abca9441,136.8,493.85,0.0,900,0,0,0,0,0,1,0.09582477754962354
1920-08-21,null,Mr.,null,null,Cochituate,Massachusetts,Middlesex County,null,0,42.28831634372727,-71.30992405399289,123160.4,1023238.6,858253,0,1,0,0,0,0,0,0,0,1,0,1,0,1,6d241af0-a1b5-6223-1843-b44e39b85659,2022-10-10T01:50:33Z,2022-10-10T02:05:33Z,1,0006bcbf-7d8c-2dcf-9c01-31cba06578bd,85.55,234.71,187.76,900,1,0,0,0,0,0,102.13552361396304
1960-03-31,null,Mr.,null,null,Gloucester,Massachusetts,Essex County,25009.0,1930,42.60004264570648,-70.66116465835674,27815.18,944263.9,19864,0,1,0,0,0,0,0,0,0,1,0,1,0,1,e1810300-6248-ae6c-d847-4d3a09fdfddb,2019-11-14T16:23:42Z,2019-11-14T17:16:28Z,1,0018bc8d-1712-9426-4839-95addc260c48,85.55,8251.87,8216.87,3166,0,0,0,0,1,0,59.62217659137577
1915-01-17,null,Mr.,null,null,Randolph,Massachusetts,Norfolk County,25021.0,2368,42.15366529705856,-71.04899600973981,221553.61,758519.43,45887,0,1,0,0,0,0,0,0,0,1,0,1,0,1,a8992104-8997-4c74-3480-3b66fd23df64,2017-08-27T14:50:54Z,2017-08-27T15:37:43Z,1,001b4d2a-a50b-c76d-d8cc-defa5a6b78e8,85.55,1179.74,943.78,2809,0,0,0,0,1,0,102.60917180013689
1960-10-12,2020-11-06,Mr.,null,null,Fall River,Massachusetts,Bristol County,25005.0,2747,41.750834597751016,-71.0452961077422,162261.25,895558.02,30440,0,0,1,0,0,0,0,0,0,1,0,1,0,1,5646f81d-85f2-c2d2-aa1d-3c30a0413148,1964-03-18T03:33:13Z,1964-03-18T03:48:13Z,0,0025e9ef-e36b-ebbe-4b19-72aa70f9a5f6,136.8,714.45,0.0,900,0,0,0,0,0,1,3.430527036276523
1959-07-01,null,Mr.,null,null,Newburyport,Massachusetts,Essex County,25009.0,1950,42.77880086615183,-70.86912980298715,95258.37,47651.28,126121,0,1,0,0,0,0,0,0,0,1,0,1,0,1,bb559f9e-c1ca-8dae-b3d5-12d95505fb98,1977-08-24T11:58:21Z,1977-08-24T12:42:44Z,0,002ab918-4e72-3216-cff6-893b003b978b,136.8,704.2,0.0,2663,0,0,0,0,0,1,18.1492128678987
1967-10-01,null,Ms.,null,null,Chelsea,Massachusetts,Suffolk County,25025.0,2151,42.4199237058549,-71.02552900372218,29100.6,1899452.29,18605,0,0,1,0,0,0,0,0,0,1,1,0,1,0,4c341ab3-7ecd-3ee6-ab0f-3fb78214d75f,2021-06-07T08:05:41Z,2021-06-07T09:17:24Z,0,002fce7f-eb35-996c-322c-7b587d8cfe2b,85.55,516.95,466.95,4303,1,0,0,0,0,0,53.68377823408624
1942-02-15,null,Mr.,null,null,Hopedale,Massachusetts,Worcester County,25027.0,1747,42.07755232349039,-71.58019896186094,111381.26,67109.96,830656,1,0,0,0,0,0,0,0,0,1,0,1,0,1,f86f92bf-9a5c-2c4e-906d-fe231585d560,2002-11-24T11:35:56Z,2002-11-24T12:23:38Z,0,0034de1c-5e1a-080a-81d1-599aa4443606,136.8,853.36,0.0,2862,0,0,0,0,0,1,60.772073921971256
1961-01-15,null,Mrs.,null,Nikolaus26,Pocasset,Massachusetts,Barnstable County,25001.0,2559,41.718558248476434,-70.57950414071792,595537.14,1067186.94,59555,1,0,0,0,0,0,0,0,0,1,0,1,1,0,377d69fa-d5e4-085a-cbda-6fabb1fb536e,1962-03-18T13:40:35Z,1962-03-18T13:55:35Z,0,00391881-6f97-9c35-d1d7-fe52b40e78f7,85.55,85.55,0.0,900,0,

#### EXTRA: Going further in feature engineering with Databricks Feature store

In this demo, we simply created a table to save our Features. Databricks offers more advanced capabilities through the use of Feature Store including collaboration, discoverabilities and realtime backend.

For more details, open [04.6-EXTRA-Feature-Store-ML-patient-readmission]($./04.6-EXTRA-Feature-Store-ML-patient-readmission).

*If you're starting your Data Science journey with Databricks, we recommend skipping this step and revisiting Databricks Feature Store later.*


## Next steps: AutoML

Data processing for data Modeling and Machine learning is simple leveraging the lakehouse. Our features are now saved as a new table or alternatively as Feature Store Tables (see Feature store notebook for more details).

We can now leverage Databricks AutoML to test multiple algorithms and generate the model training notebook including best practices. 

Open [04.2-AutoML-patient-admission-risk]($./04.2-AutoML-patient-admission-risk) to start your AutoML run.